# R version of Proximity Analysis course

In [ ]:
library(here)
library(sf)
library(tidyverse)
library(leaflet)
library(leaflet.extras)

display_map <- function(m) {
  temp_html_file <- tempfile(fileext = ".html")
  htmlwidgets::saveWidget(m, temp_html_file, selfcontained = TRUE)
  browseURL(temp_html_file)
}


In [ ]:
releases = read_sf(here("data/toxic_release_pennsylvania/toxic_release_pennsylvania/toxic_release_pennsylvania.shp")) 
stations = read_sf(here("data/PhillyHealth_Air_Monitoring_Stations/PhillyHealth_Air_Monitoring_Stations/PhillyHealth_Air_Monitoring_Stations.shp"))

In [ ]:
print(st_crs(releases))
print(st_crs(stations))

In [ ]:
recent_release <- releases[361,]
recent_release

In [ ]:
distances <- st_distance(st_geometry(recent_release), st_geometry(stations))

In [ ]:
mean(distances)

In [ ]:
print(stations[which(distances == min(distances)),][c("ADDRESS", "LONGITUDE", "LATITUDE")])

In [ ]:
two_mile_buffer <- st_buffer(stations, 2*5280)
head(two_mile_buffer$geometry)

In [ ]:
m <- leaflet() %>% 
  addTiles() %>% 
  setView(lng = -75.1652, lat = 39.9526,  zoom = 11) %>% 
  addHeatmap(data = releases,
    lng = ~LONGITUDE, 
    lat = ~LATITUDE,
    radius = 15,
    blur = 15, 
    max = 1,
    minOpacity = 0.5
  ) %>% 
  addMarkers(data = stations,
    lng = ~LONGITUDE,
    lat = ~LATITUDE
  ) %>% 
  addPolygons(data = st_transform(two_mile_buffer, 4326))

display_map(m)

In [ ]:
my_union <- st_union(two_mile_buffer)
my_union

In [ ]:
m <- leaflet() %>% 
  addTiles() %>% 
  setView(lng = -75.1652, lat = 39.9526,  zoom = 11) %>% 
  addPolygons(data = st_transform(my_union, 4326))

display_map(m)

st_contains() ans similar return an integer vector of the geometries y contained in geometry X
To get true false values pass sparse = FALSE, which returns a matrix of true false values

In [ ]:
st_contains(my_union, st_geometry(releases[361,]))

In [ ]:
# The closest station is less than two miles away
st_contains(my_union, st_geometry(releases[361,]), sparse = FALSE)[1,1]

In [ ]:
# The closest station is more than two miles away
st_contains(my_union, st_geometry(releases[359,]), sparse = FALSE)[1,1]

# Exercises

In [ ]:
collisions <- read_sf(here("data/NYPD_Motor_Vehicle_Collisions/NYPD_Motor_Vehicle_Collisions/NYPD_Motor_Vehicle_Collisions.shp"))

In [ ]:
hospitals <- read_sf(here("data/nyu_2451_34494/nyu_2451_34494/nyu_2451_34494.shp"))

create 10 kilometer buffer around each hospital

In [ ]:
tenk_buffer <- st_buffer(hospitals, 10*1000)
tenk_buffer_union <- st_union(tenk_buffer)

In [ ]:
m <- leaflet() %>% 
  addTiles() %>% 
  setView(lng = -74, lat = 40.7,  zoom = 11) %>%
  addPolygons(data = st_transform(tenk_buffer_union, 4326))

display_map(m)

In [ ]:
outside_range <- st_filter(collisions, tenk_buffer_union, .predicate = st_disjoint)

In [ ]:
nrow(outside_range)

Other ways to get points outside a union.
st_contains() output is different that the other functions. It's dense matrix is one row with many boolean columns, rather than one column with many rows.

In [ ]:
# indices <- lengths(st_within(st_geometry(collisions), tenk_buffer_union)) == 0
# indices <- !st_within(st_geometry(collisions), tenk_buffer_union, sparse = FALSE)[,1]
indices <- !st_contains(tenk_buffer_union, st_geometry(collisions), sparse = FALSE)[1,]
points_outside <- collisions[indices, ]

In [ ]:
perc_outside <- round(100*nrow(outside_range)/nrow(collisions), 2)
print(str_glue("Percentage of collisions more than 10 km away from the closest hospital: {perc_outside}%"))

In [ ]:

best_hospital <- function(collision_location) {
  distances <- st_distance(st_geometry(collision_location), st_geometry(hospitals))
  my_hospital <- hospitals[which(distances == min(distances)), ]["name"] %>% 
    st_drop_geometry() %>% 
    as.character()
  return(my_hospital)
}

best_hospital(st_geometry(outside_range[1,]))

### 5) Which hospital is under the highest demand?

Considering only collisions in the `outside_range` DataFrame, which hospital is most recommended?  

Your answer should be a Python string that exactly matches the name of the hospital returned by the function you created in **4)**.


## R vs Python

The python way is to iterate using the defined `best_hospital` function. The works terribly in R on the 40,000 row `outside_range` data.
The R way is to is to make use of already vectorized functions in the sf library: use `st_nearest_feature` to calculate all the distances at once, rather than iterating and running `st_distance` 40,000 times.

In [ ]:
# For each collision, get the index of the nearest hospital
nearest_idx <- st_nearest_feature(outside_range, hospitals)

# Pull the hospital name for each collision
best_hospital_vec <- hospitals$name[nearest_idx]

# Find the hospital closest to the most collisions
hospital_counts <- table(best_hospital_vec)
names(hospital_counts)[which.max(hospital_counts)]